# MARGIN: nnU-Net segmentation inference

Standard nnU-Net v2 inference using `fold_all/checkpoint_best.pth`. This notebook invokes the accompanying Python entry point in a subprocess, so nnU-Net multiprocessing runs in a normal script. **No inference execution or performance validation is claimed for this release.**

Download and extract the model package linked in [Model details](../docs/segmentation-model.md). Use a Python 3.10–3.12 environment and install `segmentation/requirements.txt` before selecting its Jupyter kernel. Only load checkpoints from a trusted source.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Run from the repository root or the segmentation/ directory.
ROOT = Path.cwd()
if not (ROOT / "segmentation" / "inference.py").is_file():
    ROOT = ROOT.parent
SCRIPT = ROOT / "segmentation" / "inference.py"
assert SCRIPT.is_file(), "Open this notebook from the cloned MARGIN repository."
print("Python:", sys.executable)
print("Inference entry point:", SCRIPT)

## 1. Configure local paths

`MODEL_DIR` must contain `dataset.json`, `plans.json`, and `fold_all/checkpoint_best.pth`. Each input is a 3D CT in Hounsfield units, named `CASE_0000.nii.gz`. Preserve the original NIfTI geometry; nnU-Net performs normalization, resampling, and output geometry restoration using the bundled plans. Do not manually normalize CT values or resize images to the patch size.

In [ ]:
MODEL_DIR = Path("/absolute/path/to/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres")
INPUT_DIR = Path("/absolute/path/to/imagesTs")
OUTPUT_DIR = Path("/absolute/path/to/predictions")
DEVICE = "cuda"  # Change to "cpu" if no NVIDIA GPU is available (slower).
DISABLE_TTA = False
SAVE_PROBABILITIES = False

## 2. Run inference

Default: 50% sliding-window step, Gaussian weighting, mirroring enabled, one preprocessing/export worker. Existing predictions are overwritten; add `--continue-prediction` to the command to skip them. The model folder is loaded directly, so `nnUNet_results` does not need to be configured.

In [ ]:
command = [sys.executable, str(SCRIPT),
           "--model-dir", str(MODEL_DIR),
           "--input-dir", str(INPUT_DIR),
           "--output-dir", str(OUTPUT_DIR),
           "--device", DEVICE]
if DISABLE_TTA:
    command.append("--disable-tta")
if SAVE_PROBABILITIES:
    command.append("--save-probabilities")
subprocess.run(command, check=True)

## 3. Inspect output geometry and labels

Output labels: **0** background, **1** biliary label excluding voxels assigned to tumor, **2** tumor. Training used overlapping regions: `danguan = [1, 2]` and `tumor = 2`, with `regions_class_order = [1, 2]`. Thus the training `danguan` region is recovered with `mask > 0`, while `mask == 1` isolates the non-tumor biliary label. Probability channels correspond to these two regions, not a three-class softmax. Review masks before downstream analysis.

In [ ]:
import nibabel as nib
import numpy as np

inputs = sorted(INPUT_DIR.glob("*_0000.nii.gz"))
assert inputs, "No input CT files found."
for source in inputs:
    target = OUTPUT_DIR / (source.name[:-len("_0000.nii.gz")] + ".nii.gz")
    ct, mask = nib.load(source), nib.load(target)
    values = np.unique(np.asanyarray(mask.dataobj))
    assert ct.shape == mask.shape, f"Shape mismatch: {target.name}"
    assert np.allclose(ct.affine, mask.affine, atol=1e-4), f"Affine mismatch: {target.name}"
    assert np.isin(values, [0, 1, 2]).all(), f"Unexpected labels: {values}"
    print(target.name, "shape:", mask.shape, "labels:", values.tolist())